# Actividad 2 — Principios SOLID

Para esta actividad, se mantuvo el ejemplo del Restaurante de la Actividad 01 donde se explicó como modelar un restaurante aplicando los cuatro pilares de la POO. En este caso, se reutilizó el concepto para modelar el diseño de las clases de manera que sean fáciles de mantener, extender y modificar.

## Principio de Responsabilidad Única (Single Responsibility Principle, SRP)

### Ejemplo sin SRP ❌

Para este ejemplo, la clase Restaurant se encarga de demasiadas responsabilidades:

* administrar el restaurante.
* preparar pedidos.
* guardar pedidos.
* enviar correos.
* calcular facturas.

In [1]:
class Restaurant:
    def __init__(self, name: str, orders: list, is_kitchen_open: bool, email: str) -> None:
        self.name = name
        self.orders = orders
        self.is_kitchen_open = is_kitchen_open
        self.email = email

    def prepare_order(self, order_id: str) -> None:
        if self.is_kitchen_open:
            print(f"Preparando pedido {order_id}")
        return "La cocina está cerrada, no se puede preparar el pedido."

    def save_order(self, order_id: str) -> None:
        self.orders.append(order_id)
        print(f"Pedido {order_id} guardado en la base de datos.")

    def send_confirmation(self) -> None:
        print(f"Enviando confirmación a {self.email}")

    def calculate_bill(self, subtotal: float) -> float:
        return subtotal * 1.19

In [2]:
restaurant = Restaurant("Pasta Palace", [], True, "pasta@example.com")
restaurant.prepare_order("001")
restaurant.save_order("001")
restaurant.send_confirmation()
print(restaurant.calculate_bill(100))

Preparando pedido 001
Pedido 001 guardado en la base de datos.
Enviando confirmación a pasta@example.com
119.0


### Ejemplo con SRP ✅

Ahora, la clase Restaurant solo se encarga de la preparación de los pedidos mientras que las clases OrderRepository y InvoiceService se encargan de gestionar la persistencia de los pedidos y el cálculo de las facturas respectivamente.

Por lo que:

* Restaurant cambia si cambia la lógica de preparación.
* OrderRepository cambia si cambia la persistencia.
* InvoiceService cambia si cambia la lógica de facturación.

In [3]:
class Restaurant:
    def __init__(self, name: str, is_kitchen_open: bool) -> None:
        self.name = name
        self.is_kitchen_open = is_kitchen_open

    def prepare_order(self, order_id: str) -> str:
        if self.is_kitchen_open:
            return f"Preparando pedido {order_id}"
        return "La cocina está cerrada, no se puede preparar el pedido."

    def open_kitchen(self):
        self.is_kitchen_open = True


class OrderRepository:
    def __init__(self, database_name: str, orders: list) -> None:
        self.database_name = database_name
        self.orders = orders

    def save_order(self, order_id: str) -> str:
        self.orders.append(order_id)
        return f"Pedido {order_id} guardado en {self.database_name}"

    def delete_order(self, order_id: str) -> None:
        self.orders.remove(order_id)

    def get_orders(self) -> list:
        return self.orders

class InvoiceService:
    def __init__(self, tax_rate: float, currency: str) -> None:
        self.tax_rate = tax_rate
        self.currency = currency

    def calculate_total(self, subtotal: float) -> float:
        return subtotal + subtotal * self.tax_rate

    def format_total(self, subtotal: float) -> str:
        total = self.calculate_total(subtotal)
        return f"{self.currency} {total:.2f}"


In [4]:
restaurant = Restaurant("Pasta Palace", True)
print(restaurant.prepare_order("001"))

order_repo = OrderRepository("orders_db", [])
print(order_repo.save_order("001"))
print(order_repo.save_order("002"))
order_repo.delete_order("001")
print(order_repo.get_orders())

invoice_service = InvoiceService(0.19, "USD")
print(invoice_service.format_total(100))

Preparando pedido 001
Pedido 001 guardado en orders_db
Pedido 002 guardado en orders_db
['002']
USD 119.00


## Principio Abierto/Cerrado (Open/Closed Principle, OCP)

Para este ejemplo, tomamos el dominio de los Productos del restaurante para aplicar este principio. Como el restaurante aplica diferentes impuestos para cada tipo de producto (Food, Beverage y Dessert), qué pasaría en caso se agregue un nuevo tipo de producto llamado Snacks o Extras?

### Ejemplo sin OCP ❌

La clase PriceCalculator calcula el precio de cada tipo de producto aplicando un impuesto por tipo. El problema ocurre al querer agregar el nuevo tipo de producto puesto que se debe modificar inevitablemente la clase.

In [5]:
class PriceCalculator:
    def __init__(self, tax_rate: float, currency: str) -> None:
        self.tax_rate = tax_rate
        self.currency = currency

    def calculate(self, product_type: str, price: float) -> float:
        if product_type == "food":
            return price * 1.10
        elif product_type == "beverage":
            return price * 1.03
        elif product_type == "dessert":
            return price * 1.05
        else:
            return price * (1 + self.tax_rate)

    def format_price(self, price: float) -> str:
        return f"{self.currency} {price:.2f}"

In [6]:
price_calculator = PriceCalculator(0.19, "COP")
food = price_calculator.calculate("food", 20000)
beverage = price_calculator.calculate("beverage", 8000)
dessert = price_calculator.calculate("dessert", 12000)

print(price_calculator.format_price(food))
print(price_calculator.format_price(beverage))
print(price_calculator.format_price(dessert))



COP 22000.00
COP 8240.00
COP 12600.00


### Ejemplo con OCP ✅

Ahora, siguiendo con el ejemplo mostrado en el ejercicio del Polimorfismo de la actividad pasada, cada tipo de producto (clases hijas) se encargan de calcular sus precios e impuestos sin modificar o comprometer el funcionamiento de las demás.

In [7]:
from abc import ABC, abstractmethod


class Product(ABC):
    def __init__(self, name: str, price: float) -> None:
        self.name = name
        self.price = price

    @abstractmethod
    def get_final_price(self) -> float:
        pass

    def show_name(self) -> str:
        return self.name


class Food(Product):
    def get_final_price(self) -> float:
        return self.price * 1.10

    def get_category(self) -> str:
        return "Food"


class Beverage(Product):
    def get_final_price(self) -> float:
        return self.price * 1.03

    def get_category(self) -> str:
        return "Beverage"


class Dessert(Product):
    def get_final_price(self) -> float:
        return self.price * 1.05

    def get_category(self) -> str:
        return "Dessert"

In [8]:
products = [ Food("Burger", 20000), Beverage("Juice", 8000), Dessert("Cake", 12000)]

for product in products:
    print(f"{product.show_name()} ({product.get_category()}): {product.get_final_price()}")


Burger (Food): 22000.0
Juice (Beverage): 8240.0
Cake (Dessert): 12600.0


Ahora podremos agregar el nuevo tipo de producto, sin ningún problema:

In [9]:
class Snacks(Product):
    def get_final_price(self) -> float:
        return self.price # Al ser un snack, no se aplica ningún impuesto adicional, por lo que el precio final es el mismo que el precio base.

    def get_category(self) -> str:
        return "Snacks"

products.append(Snacks("Chips", 5000))

for product in products:
    print(f"{product.show_name()} ({product.get_category()}): {product.get_final_price()}")

Burger (Food): 22000.0
Juice (Beverage): 8240.0
Cake (Dessert): 12600.0
Chips (Snacks): 5000


## Principio de Sustitución de Liskov (Liskov Substitution Principle)

Expliquemos este principio centrandonos ahora en una clase Empleado

### Ejemplo sin LSP ❌

La clase de empleado tiene un método `take_order` que puede ejecutar perfectamente un mesero pero no un cocinero, es decir:

Empleado -> toma pedidos

Cocinero -> es un empleado

Cocinero -> NO puede tomar pedidos

In [10]:
class Employee:
    def __init__(self, name: str, position: str) -> None:
        self.name = name
        self.position = position

    def take_order(self, pedido: str) -> str:
        return f"Empleado {self.name} ( {self.position} ) tomando pedido: {pedido}"

    def work(self) -> str:
        return f"{self.name} trabajando como {self.position}"

class Waiter(Employee):
    def __init__(self, name: str, position: str, table_area: str) -> None:
        super().__init__(name, position)
        self.table_area = table_area

    def work(self) -> str:
        return f"{self.name} atendiendo {self.table_area}"

class Chef(Employee):
    def __init__(self, name: str, position: str, specialty: str) -> None:
        super().__init__(name, position)
        self.specialty = specialty

    def take_order(self, pedido: str) -> None:
        raise NotImplementedError("Un cocinero no puede tomar pedidos")

    def work(self) -> str:
        return f"Cocinero {self.name} cocinando {self.specialty}"

In [11]:
mesero = Waiter(name="Juan", position="Mesero", table_area="Área Principal")
cocinero = Chef(name="Pedro", position="Cocinero", specialty="Pizzas")

print(mesero.take_order("Hamburguesa"))
print(cocinero.take_order("Hamburguesa"))

Empleado Juan ( Mesero ) tomando pedido: Hamburguesa


NotImplementedError: Un cocinero no puede tomar pedidos

### Ejemplo con LSP ✅


La solución es definir en la clase Employee únicamente comportamientos que todo empleado pueda cumplir como es el caso del método `work`.

In [ ]:
class Employee:
    def __init__(self, name: str, position: str) -> None:
        self.name = name
        self.position = position

    def work(self) -> str:
        return f"Empleado {self.name} trabajando como {self.position}"

class Waiter(Employee):
    def __init__(self, name: str, position: str, table_area: str = "Área Principal") -> None:
        super().__init__(name, position)
        self.table_area = table_area

    def take_order(self, pedido: str) -> str:
        return f"Mesero {self.name} tomando pedido: {pedido}"

    def work(self) -> str:
        return super().work() + f" atendiendo mesas"


class Chef(Employee):
    def __init__(self, name: str, position: str, specialty: str = "Cocina general") -> None:
        super().__init__(name, position)
        self.specialty = specialty

    def cook(self, dish: str) -> str:
        return f"Cocinero {self.name} cocinando: {dish}"

    def work(self) -> str:
        return super().work() + f" preparando comida"

In [ ]:
employees = [Waiter("Juan", "Mesero"), Chef("Pedro", "Cocinero"), Employee("Ana", "Recepcionista")]

for employee in employees:
    print(employee.work())

Empleado Juan trabajando como Mesero atendiendo mesas
Empleado Pedro trabajando como Cocinero preparando comida
Empleado Ana trabajando como Recepcionista


## Principio de Segregación de Interfaces (Interface Segregation Principle, ISP)

### Ejemplo sin ISP ❌

En este ejemplo, la clase Employee define 4 métodos abstractos que hereda su clase hija Chef pero solo uno de ellos es verdaderamente relevante: función `cook`.

In [ ]:
from abc import ABC, abstractmethod


class Employee(ABC):
    @abstractmethod
    def cook(self) -> str:
        pass

    @abstractmethod
    def take_order(self) -> str:
        pass

    @abstractmethod
    def deliver_order(self) -> str:
        pass


class Chef(Employee):
    def __init__(self, name: str, specialty: str) -> None:
        self.name = name
        self.specialty = specialty

    def cook(self) -> str:
        return f"{self.name} está cocinando {self.specialty}."

    def take_order(self) -> str:
        raise NotImplementedError("Un cocinero no puede tomar pedidos")

    def deliver_order(self) -> str:
        raise NotImplementedError("Un cocinero no puede entregar pedidos")

In [ ]:
chef = Chef(name="Pedro", specialty="Pasta")

print(chef.cook())

Pedro está cocinando Pasta.


In [ ]:
print(chef.take_order())

NotImplementedError: Un cocinero no puede tomar pedidos

In [ ]:
print(chef.deliver_order())

NotImplementedError: Un cocinero no puede entregar pedidos

### Ejemplo con ISP ✅

Aplicando el principio de segregación de interfaces, se crean 3 interfaces que puedan ser de utilidad para cada tipo de empleado. Asi:

* Un chef puede implementar la interfaz `Cookable`.
* Un mesero puede implementar la interfaz `OrderTaker`.
* Un repartidor puede implementar la interfaz `Deliverable`.

In [ ]:
from abc import ABC, abstractmethod


class Cookable(ABC):

    @abstractmethod
    def cook(self) -> str:
        pass

    @abstractmethod
    def kitchen_info(self) -> str:
        pass


class OrderTaker(ABC):

    @abstractmethod
    def take_order(self) -> str:
        pass

    @abstractmethod
    def register_customer(self, name: str) -> str:
        pass


class Deliverable(ABC):

    @abstractmethod
    def deliver_order(self) -> str:
        pass

    @abstractmethod
    def delivery_zone(self) -> str:
        pass


class Chef(Cookable):
    def __init__(self, name: str, specialty: str) -> None:
        self.name = name
        self.specialty = specialty

    def cook(self) -> str:
        return f"{self.name} está cocinando {self.specialty}."

    def kitchen_info(self) -> str:
        return f"Especialidad de {self.name}: {self.specialty}"


class Waiter(OrderTaker):
    def __init__(self, name: str, table_area: str) -> None:
        self.name = name
        self.table_area = table_area

    def take_order(self) -> str:
        return f"{self.name} está tomando una orden."

    def register_customer(self, name: str) -> str:
        return f"{name} registrado por {self.name}."


class DeliveryDriver(Deliverable):
    def __init__(self, name: str, zone: str) -> None:
        self.name = name
        self.zone = zone

    def deliver_order(self) -> str:
        return f"{self.name} está entregando el pedido."

    def delivery_zone(self) -> str:
        return f"Zona de entrega: {self.zone}"


In [ ]:
chef = Chef("Laura", "comida italiana")
waiter = Waiter("Andrés", "salón principal")
driver = DeliveryDriver("Carlos", "Zona Norte")

print(chef.cook())
print(chef.kitchen_info())

print(waiter.take_order())
print(waiter.register_customer("María"))

print(driver.deliver_order())
print(driver.delivery_zone())


Laura está cocinando comida italiana.
Especialidad de Laura: comida italiana
Andrés está tomando una orden.
María registrado por Andrés.
Carlos está entregando el pedido.
Zona de entrega: Zona Norte


## Principio de Inversión de Dependencias (Dependency Inversion Principle, DIP)

### Ejemplo sin DIP ❌

En este ejemplo, creamos una clase de alto nivel KitchenDisplay que se encarga de emitir la informacion de los pedidos tomados por los cocineros. Dicha clase depende de la clase ThermalPrinter (Dispositivo físico).

Supongamos que el restaurante desea instalar una pantalla digital para mayor comodidad. Lamentablemente, la clase KitchenDisplay se rompe y debe ser modificada internamente.

In [ ]:
class ThermalPrinter:
    def __init__(self, paper_width_mm: int, paper_height_mm: int) -> None:
        self.paper_width_mm = paper_width_mm
        self.paper_height_mm = paper_height_mm

    def print_ticket(self, order_text: str) -> str:
        return f"[Impresora Térmica] Imprimiendo ticket: {order_text}"

    def get_paper_info(self) -> str:
        return f"Ancho de papel: {self.paper_width_mm} mm, Alto de papel: {self.paper_height_mm} mm"


class KitchenDisplay:
    def __init__(self, station_name: str, thermal_printer: ThermalPrinter) -> None:
        self.station_name = station_name
        self.printer = thermal_printer

    def process_order(self, order_id: str, items: list) -> str:
        order_details = f"{self.station_name} - Pedido #{order_id}: {', '.join(items)}"
        return self.printer.print_ticket(order_details)

    def check_printer_health(self) -> str:
        return self.printer.get_paper_info()

In [ ]:
printer = ThermalPrinter(80, 200)
kitchen_display = KitchenDisplay("Estación de Cocina", printer)

orders = ["Pizza", "Ensalada","Torta de queso"]
print(kitchen_display.process_order("001", orders))
print(kitchen_display.check_printer_health())

[Impresora Térmica] Imprimiendo ticket: Estación de Cocina - Pedido #001: Pizza, Ensalada, Torta de queso
Ancho de papel: 80 mm, Alto de papel: 200 mm


### Ejemplo con DIP ✅

Para solucionarlo, se define una interfaz OrderPrinterInterface y como consecuencia, la clase de alto nivel KitchenDisplay solo depende únicamente de ella agregando la posibilidad de inyectar cualquier otro sistema de emisión de órdenes.

In [13]:
from abc import ABC, abstractmethod


class OrderPrinterInterface(ABC):
    @abstractmethod
    def send_to_kitchen(self, order_text: str) -> str:
        pass

    @abstractmethod
    def get_device_status(self) -> str:
        pass


class ThermalPrinter(OrderPrinterInterface):
    def __init__(self, paper_width_mm: int, paper_height_mm: int) -> None:
        self.paper_width_mm = paper_width_mm
        self.paper_height_mm = paper_height_mm

    def send_to_kitchen(self, order_text: str) -> str:
        return f"[Impresora Térmica] Imprimiendo ticket: {order_text}"

    def get_device_status(self) -> str:
        return f"[Impresora Térmica] Ancho de papel: {self.paper_width_mm} mm, Alto de papel: {self.paper_height_mm} mm"


class DigitalScreen(OrderPrinterInterface):
    def __init__(self, screen_id: str, ip_address: str) -> None:
        self.screen_id = screen_id
        self.ip_address = ip_address

    def send_to_kitchen(self, order_text: str) -> str:
        return f"[Pantalla Digital ID {self.screen_id} @ {self.ip_address}] Notificación en pantalla: {order_text}"

    def get_device_status(self) -> str:
        return f"[Pantalla Digital ID {self.screen_id} @ {self.ip_address}]: Conexión red OK."


class KitchenDisplay:
    def __init__(self, station_name: str, output_device: OrderPrinterInterface) -> None:
        self.station_name = station_name
        self.device = output_device

    def process_order(self, order_id: str, items: list) -> str:
        order_details = f"{self.station_name} - Pedido #{order_id}: {', '.join(items)}"
        return self.device.send_to_kitchen(order_details)

    def check_device_health(self) -> str:
        return self.device.get_device_status()

In [14]:
printer = ThermalPrinter(80, 200)
kitchen_display = KitchenDisplay("Estación de Cocina", printer)
orders = ["Pizza", "Ensalada","Torta de queso"]

print(kitchen_display.process_order("001", orders))
print(kitchen_display.check_device_health())

digital_screen = DigitalScreen("DS-01", "192.168.1.100")
kitchen_display = KitchenDisplay("Estación de Cocina", digital_screen)
print(kitchen_display.process_order("001", orders))
print(kitchen_display.check_device_health())

[Impresora Térmica] Imprimiendo ticket: Estación de Cocina - Pedido #001: Pizza, Ensalada, Torta de queso
[Impresora Térmica] Ancho de papel: 80 mm, Alto de papel: 200 mm
[Pantalla Digital ID DS-01 @ 192.168.1.100] Notificación en pantalla: Estación de Cocina - Pedido #001: Pizza, Ensalada, Torta de queso
[Pantalla Digital ID DS-01 @ 192.168.1.100]: Conexión red OK.
